# 🗣️ 文本数据处理基础专项 (Text Processing & Sets)

本模块专门补齐在进行 NLP (自然语言处理) 或日常文本清洗时，常见的 Python 字符串操作、集合运算 (Sets)、正则表达式 (Regex) 以及 Pandas `.str` 的基础知识盲区。

## 📚 数据集说明
优先使用真实数据集：`nicapotato/womens-ecommerce-clothing-reviews` (23K 条电商评论，含评分+文本)。

In [1]:
# ======== 模块 0: 数据准备 & 函数加油站 ========
# 强制使用绝对路径调用 kaggle，下载数据并解压
!kaggle datasets download nicapotato/womens-ecommerce-clothing-reviews --path ./data --unzip -f "Womens Clothing E-Commerce Reviews.csv"

import pandas as pd
import numpy as np
import re

# 加载数据
df = pd.read_csv('./data/Womens Clothing E-Commerce Reviews.csv', index_col=0)
display(df.head(3))

./Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Dataset URL: https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews
License(s): CC0-1.0
  0%|                                               | 0.00/2.79M [00:00<?, ?B/s]
100%|██████████████████████████████████████| 2.79M/2.79M [00:00<00:00, 2.97GB/s]


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses


## ⛽ 函数加油站 (Function Cheat Sheet)

在练习开始前，先熟悉今日核心函数：

| 函数名 | 大白话解释 | 标准语法与常用参数 | SQL 类比 |
| :--- | :--- | :--- | :--- |
| `set.union()` 或 `|` | **合并去重**：把两个集合合在一起并自动去重 | `set1.union(set2)` 或 `set1 | set2` | `UNION` (非 UNION ALL) |
| `set.intersection()` 或 `&`| **取交集**：找出两个集合中都有的元素 | `set1 & set2` | `INNER JOIN` (取主键交集) |
| `set.difference()` 或 `-` | **取差集**：找出一个集合有而另一个没有的元素 | `set1 - set2` | `LEFT JOIN ... WHERE B.id IS NULL` |
| `str.join()` | **胶水拼接**：把一个列表里的词用特定字符粘起来 | `" ".join(['我', '爱', '学习'])` -> `"我 爱 学习"` | `STRING_AGG()` 或 `GROUP_CONCAT()` |
| `re.sub()` | **正则替换**：按规则查找并替换 (去噪神器) | `re.sub(r'[^a-zA-Z]', ' ', text)` | `REGEXP_REPLACE()` |
| `.str.contains()` | **文本包含检查**：Pandas 里的模糊匹配 | `df['col'].str.contains('keyword', na=False, case=False)` | `LIKE '%keyword%'` |

## 🛠️ Level 1: Python 原生基础 (集合与字符串)

**场景**: 在统计词频或者特征提取时，经常需要对一组组的 Tag 或 Word 进行去重和比对。
请对以下两个用户购买的商品标签列表进行集合运算。

In [6]:
# Level 1 练习环境
user_a_tags = ['Dress', 'Shirt', 'Jeans', 'Dress', 'Shoes']
user_b_tags = ['Shoes', 'Hat', 'Jeans', 'Sunglasses']

# Q1. 将列表转换为 Set 集合 (以此达到自身去重的目的)
set_a = set(user_a_tags)
set_b = set(user_b_tags)
print(f"Set A (去重后): {set_a}")
print(f"Set B (去重后): {set_b}")

# Q2. 求并集 (找出两个用户购买过的所有商品种类)
union_set = set.union(set_a,set_b)
print(f"并集: {union_set}")

# Q3. 求交集 (找出俩人都买过的商品)
intersect_set = set.intersection(set_a,set_b)
print(f"交集: {intersect_set}")

# Q4. 将交集的结果用逗号拼接成一个字符串 (准备入库)
joined_str = ",".join(intersect_set)
print(f"拼接字符串: {joined_str}")

Set A (去重后): {'Shirt', 'Shoes', 'Dress', 'Jeans'}
Set B (去重后): {'Sunglasses', 'Shoes', 'Jeans', 'Hat'}
并集: {'Hat', 'Shirt', 'Shoes', 'Dress', 'Sunglasses', 'Jeans'}
交集: {'Shoes', 'Jeans'}
拼接字符串: Shoes,Jeans


In [ ]:
# ✅ Level 1 参考答案
set_a = set(user_a_tags)
set_b = set(user_b_tags)
union_set = set_a.union(set_b)  # 或 set_a | set_b
intersect_set = set_a & set_b
joined_str = ", ".join(intersect_set) # 注意交际元素的顺序在 set 中是不固定的

## 🕵️‍♂️ Level 2: Pandas `.str` 与 正则表达式 (Regex) 去噪

**场景**: 我们拿到了真实的电商评论数据 `df`，但是文本里可能有很多标点符号、全大写/全小写不统一的情况，并且存在缺失值。这会导致 NLP 词频统计失效（比如 "Dress" 和 "dress." 会被当成两个词）。

你的任务是：
1. 只保留包含 'Review Text' 的有效评论行。
2. 将所有评论转换为小写。
3. 使用正则表达式，将所有非字母数字的字符 (例如标点符号) 替换为空格。

In [13]:
# 获取一份工作副本
df_nlp = df[['Title', 'Review Text']].copy()

# Q1. 删除 Review Text 列为空的行
df_nlp = df_nlp.dropna(subset='Review Text')

# Q2. 将 Review Text 所有字母变为小写 (Tip: 使用 .str.lower())
df_nlp['clean_text'] = df_nlp['Review Text'].str.lower()

# Q3. 使用 apply 和 re.sub 替换标点符号
# Regex 提示: r'[^a-zA-Z0-9]' 表示匹配所有非字母和非数字的字符
def remove_punctuation(text):
    if pd.isna(text):
        return text
    # 在这里写替换逻辑，把非字母数字替换成空格 ' '
    return re.sub(r'[^a-zA-Z0-9]', ' ', text)

df_nlp['clean_text'] = df_nlp['clean_text'].apply(remove_punctuation)

display(df_nlp.head(3))

,Title,Review Text,clean_text
0,NaN,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky and sexy and comf...
1,NaN,Love this dress! it's sooo pretty. i happene...,love this dress it s sooo pretty i happene...
2,Some major design flaws,I had such high hopes for this dress and reall...,i had such high hopes for this dress and reall...


In [ ]:
# ✅ Level 2 参考答案
df_nlp = df[['Title', 'Review Text']].copy()
df_nlp = df_nlp.dropna(subset=['Review Text'])
df_nlp['clean_text'] = df_nlp['Review Text'].str.lower()

def remove_punctuation(text):
    if pd.isna(text):
        return text
    return re.sub(r'[^a-zA-Z0-9]', ' ', text)

df_nlp['clean_text'] = df_nlp['clean_text'].apply(remove_punctuation)
# Pandas 一行流替代方案：df_nlp['clean_text'] = df_nlp['clean_text'].str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)


## 🚀 Level 3: 复杂业务场景 - 拼装评论上下文

**场景**: 用户的评价通常包含 `Title` (标题) 和 `Review Text` (正文)。有时用户只写了正文没写标题，有时都写了。在喂给算法模型（如 TF-IDF 或 LDA）前，我们希望把它们**拼接成一个完整长文本**。

如果用 `df['Title'] + " " + df['Review Text']`，只要其中一个为 NaN，结果就会是 NaN （Pandas 的特性）。

**任务**: 优雅地将这两列合二为一，缺失部分当做空字符串处理，去除两端多余空格。

In [16]:
# 构造包含缺失值的测试用例
df_test = pd.DataFrame({
    'Title': ['Great Dress!', np.nan, 'Too small'],
    'Review Text': ['I love it', 'Runs large and color is off.', np.nan]
})
display(df_test)

# Q1. 先将 NaN 填充为空字符串 ""
df_test['Title_Fill'] = df_test['Title'].fillna("")
df_test['Review_Fill'] = df_test['Review Text'].fillna("")

# Q2. 拼接两列，并在中间加一个空格
df_test['Full_Text'] = df_test['Title_Fill'] + " " + df_test['Review_Fill']

# Q3. 去除拼接后可能产生的两端多余空格 (例如只有 Title 没有 Review 时，末尾会多一个空格)
df_test['Full_Text'] = df_test['Full_Text'].str.strip()

# 查看最终结果
display(df_test[['Full_Text']])

,Title,Review Text
0,Great Dress!,I love it
1,NaN,Runs large and color is off.
2,Too small,NaN


,Full_Text
0,Great Dress! I love it
1,Runs large and color is off.
2,Too small


In [18]:
# ✅ Level 3 参考答案
df_test['Title_Fill'] = df_test['Title'].fillna("")
df_test['Review_Fill'] = df_test['Review Text'].fillna("")
df_test['Full_Text'] = df_test['Title_Fill'] + " " + df_test['Review_Fill']
df_test['Full_Text'] = df_test['Full_Text'].str.strip() # strip 去除首尾空格

# 💡 进阶：Pandas apply + join 一行流极致优雅写法
df_test['Full_Text'] = df_test[['Title', 'Review Text']].apply(lambda x: " ".join(x.dropna().astype(str)), axis=1)



